<a href="https://colab.research.google.com/github/RMoulla/GIA_Nanterre/blob/main/Gemma4_E2B_MASSIVE_FR_1000_QLoRA_T4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Apprendre à Gemma 4 à reconnaître une intention en français

Nous allons entraîner **Gemma 4 E2B** sur **1 000 exemples français de MASSIVE**.
Le modèle reçoit une demande, par exemple « Réveille-moi à sept heures », et répond avec une étiquette, ici `alarm_set`.
Il doit choisir parmi **60 intentions** : nous entraînons un classifieur, pas un assistant de conversation généraliste.

Nous utiliserons **QLoRA** : les poids de base sont chargés en 4 bits et restent gelés ; seuls de petits adaptateurs LoRA sont entraînés.
La validation sert à choisir la meilleure époque. Le test sert à comparer le modèle adapté à sa base, une fois l'entraînement terminé.

Dans Colab, choisir **Exécution → Modifier le type d'exécution → GPU T4**, puis exécuter les cellules dans l'ordre.
Le parcours est : **préparer les données → formater les réponses → entraîner → mesurer → télécharger**.


## 1. Préparer l'environnement

Nous fixons les versions principales utilisées pour cette expérience. Cette cellule installe les bibliothèques dans le Python du notebook.
Si Colab demande un redémarrage après l'installation, redémarrer la session, puis poursuivre les cellules dans l'ordre.


In [ ]:
%pip -q install "transformers==5.10.1" "peft==0.19.0" "accelerate==1.14.0" "bitsandbytes==0.50.2" "datasets==4.8.5" scikit-learn sentencepiece matplotlib


### Fixer les données, le modèle et le dossier de travail

Une **révision** désigne une version précise d'un dépôt Hugging Face. Les deux révisions ci-dessous fixent le modèle et le corpus.
La graine `42` permet de retrouver la même sélection d'exemples. Les fichiers seront enregistrés dans `/content/gemma4_massive_fr_1000`.


In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
from importlib.metadata import version
import gc
import hashlib
import json
import os
import random
import re
import time
import unicodedata

SEED = 42
MODEL_ID = "google/gemma-4-E2B-it"
MODEL_REVISION = "3e22461f65e89153144f8adb70e3b8c2cc9845a7"
DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "ed58ac423a2f4121720918bf5301577edce4ffd3"
SAMPLE_SIZES = {"train": 1000, "validation": 300, "test": 300}
RUN_DIR = Path("/content/gemma4_massive_fr_1000")
DATA_DIR = RUN_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


### Vérifier que Colab utilise bien le T4

Le GPU réalise l'entraînement. Nous utiliserons le calcul en **fp16**, adapté au T4, et nous initialisons la graine avant de créer le modèle.


In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed,
)

assert torch.cuda.is_available(), "Choisir un GPU dans les paramètres Colab."
assert "T4" in torch.cuda.get_device_name(0), "Cette expérience attend un GPU T4."
set_seed(SEED)
print("GPU :", torch.cuda.get_device_name(0))


## 2. Charger MASSIVE en français

MASSIVE fournit déjà trois ensembles : entraînement, validation et test. Nous conservons cette séparation.
Nous lisons directement leurs fichiers **Parquet**. Dans le corpus, `utt` est la phrase et `intent` est l'intention à prédire.


In [ ]:
DATA_URLS = {
    split: (
        f"https://huggingface.co/datasets/{DATASET_ID}/resolve/"
        f"{DATASET_REVISION}/fr-FR/{split}/0000.parquet"
    )
    for split in SAMPLE_SIZES
}
raw = load_dataset(
    "parquet", data_files=DATA_URLS,
    cache_dir=str(DATA_DIR / ".dataset_cache"),
)
intent_names = list(raw["train"].features["intent"].names)
assert len(intent_names) == len(set(intent_names)) == 60
print(raw)
print("Exemple :", raw["train"][0]["utt"])
print("Intention :", intent_names[raw["train"][0]["intent"]])


### Remplacer les numéros de classes par leurs noms

Les fichiers stockent les intentions sous forme de numéros. Nous utilisons leur correspondance officielle pour obtenir des noms tels que `alarm_set`.
Nous gardons aussi les identifiants : ils permettront de retrouver les exemples utilisés.


In [ ]:
original = {}
for split in SAMPLE_SIZES:
    assert list(raw[split].features["intent"].names) == intent_names
    expected_partition = "dev" if split == "validation" else split
    rows = []
    for row in raw[split]:
        assert row["locale"] == "fr-FR"
        assert row["partition"] == expected_partition
        assert isinstance(row["utt"], str) and row["utt"].strip()
        label_id = int(row["intent"])
        assert 0 <= label_id < len(intent_names)
        rows.append({
            "id": str(row["id"]), "locale": row["locale"],
            "utt": row["utt"], "intent_id": label_id,
            "intent": intent_names[label_id],
        })
    original[split] = rows

assert {split: len(rows) for split, rows in original.items()} == {
    "train": 11514, "validation": 2033, "test": 2974,
}


## 3. Éviter que le modèle revoie les phrases du test

Nous comparons les phrases après normalisation des caractères, des majuscules et des espaces.
Nous excluons de l'entraînement **toutes** les phrases présentes dans la validation ou le test officiels, avant de sélectionner nos petits échantillons.
Si une phrase apparaît en validation et en test, le test a priorité.


In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", text).casefold()
    return re.sub(r"\s+", " ", text).strip()


original_keys = {
    split: {normalize_text(row["utt"]) for row in rows}
    for split, rows in original.items()
}
blocked = {
    "test": set(),
    "validation": original_keys["test"],
    "train": original_keys["validation"] | original_keys["test"],
}


### Retirer les doublons et les annotations contradictoires

Cette fonction regroupe les phrases identiques après normalisation. Elle garde la première occurrence si les étiquettes sont identiques ; elle retire tout le groupe si les étiquettes se contredisent.
Le petit bilan retourné servira à documenter les données sauvegardées.


In [ ]:
def clean_rows(rows, blocked_texts):
    groups = defaultdict(list)
    blocked_rows = 0
    for row in rows:
        key = normalize_text(row["utt"])
        if key in blocked_texts:
            blocked_rows += 1
        else:
            groups[key].append(row)

    cleaned = []
    duplicates = conflicts = conflicting_rows = 0
    for group in groups.values():
        if len({row["intent"] for row in group}) > 1:
            conflicts += 1
            conflicting_rows += len(group)
        else:
            cleaned.append(group[0])
            duplicates += len(group) - 1

    details = {
        "input_rows": len(rows), "blocked_overlap_rows": blocked_rows,
        "duplicate_rows_removed": duplicates,
        "conflicting_text_groups_removed": conflicts,
        "conflicting_rows_removed": conflicting_rows,
        "clean_rows": len(cleaned),
    }
    return cleaned, details


## 4. Sélectionner exactement 1 000 exemples d'entraînement

Un tirage purement aléatoire pourrait oublier les intentions rares. Nous réservons **un exemple par intention présente**, puis répartissons les places restantes proportionnellement à leurs fréquences.
Les arrondis sont distribués aux plus grands restes pour obtenir exactement le nombre demandé. Nous tirons ensuite les exemples avec la graine `42`.


In [ ]:
def stratified_sample(rows, size):
    groups = defaultdict(list)
    for row in rows:
        groups[row["intent"]].append(row)
    labels = sorted(groups)
    assert len(labels) <= size <= len(rows)

    remaining = size - len(labels)
    exact = {label: remaining * len(groups[label]) / len(rows) for label in labels}
    quotas = {label: 1 + int(exact[label]) for label in labels}
    order = sorted(labels, key=lambda label: (-(exact[label] % 1), label))
    left = size - sum(quotas.values())
    for label in order:
        if left and quotas[label] < len(groups[label]):
            quotas[label] += 1
            left -= 1
    assert left == 0

    rng = random.Random(SEED)
    sample = []
    for label in labels:
        sample.extend(rng.sample(groups[label], quotas[label]))
    rng.shuffle(sample)
    assert len(sample) == size
    assert {row["intent"] for row in sample} == set(labels)
    return sample


### Appliquer la même méthode aux trois ensembles

L'entraînement contient 60 intentions. Les ensembles officiels de validation et de test n'en contiennent chacun que 59 :
`audio_volume_other` est absent de la validation, et `cooking_query` du test. Nous conservons cette propriété.


In [ ]:
selected = {}
split_details = {}
for split, size in SAMPLE_SIZES.items():
    cleaned, details = clean_rows(original[split], blocked[split])
    sample = stratified_sample(cleaned, size)
    selected[split] = sample
    details.update({
        "sample_rows": len(sample),
        "sample_intent_counts": dict(sorted(Counter(row["intent"] for row in sample).items())),
        "missing_intents": sorted(set(intent_names) - {row["intent"] for row in sample}),
        "sample_ids": [row["id"] for row in sample],
    })
    split_details[split] = details
    print(f"{split} : {len(sample)} exemples, {len(details['sample_intent_counts'])} intentions")

train_rows = selected["train"]
val_rows = selected["validation"]
test_rows = selected["test"]
assert len({row["intent"] for row in train_rows}) == 60


### Vérifier la séparation des exemples

Avant l'entraînement, nous contrôlons les tailles, l'absence de doublons internes et l'absence de phrase ou d'identifiant commun entre les trois ensembles.
Ces vérifications empêchent une évaluation artificiellement améliorée par la mémorisation d'une phrase déjà vue.


In [ ]:
selected_ids = {}
selected_texts = {}
for split, rows in selected.items():
    selected_ids[split] = {row["id"] for row in rows}
    selected_texts[split] = {normalize_text(row["utt"]) for row in rows}
    assert len(rows) == SAMPLE_SIZES[split]
    assert len(selected_ids[split]) == len(rows)
    assert len(selected_texts[split]) == len(rows)

for first, second in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert not selected_ids[first] & selected_ids[second]
    assert not selected_texts[first] & selected_texts[second]

print("Vérifications réussies : aucun exemple commun entre les trois ensembles.")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2))


### Garder une copie des données utilisées

Chaque ligne d'un fichier JSONL représente un exemple. Nous sauvegardons les trois sélections et leur description dans `data/` : une nouvelle session pourra retrouver les mêmes exemples et leur provenance.


In [ ]:
for split, rows in selected.items():
    contents = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows)
    (DATA_DIR / f"massive_fr_{split}.jsonl").write_text(contents, encoding="utf-8")
    split_details[split]["jsonl_sha256"] = hashlib.sha256(contents.encode("utf-8")).hexdigest()

audit = {
    "dataset_id": DATASET_ID, "dataset_revision": DATASET_REVISION,
    "source_urls": DATA_URLS, "locale": "fr-FR", "seed": SEED,
    "normalization": "NFKC + casefold + collapse whitespace + strip",
    "deduplication": "test priority; validation excludes all original test texts; train excludes all original validation/test texts; remove internally conflicting groups then keep first duplicate",
    "sampling": "one example per present intent plus proportional largest-remainder quotas; random.Random(42), intent names sorted",
    "intent_names": intent_names, "splits": split_details,
}
(DATA_DIR / "data_audit.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)


## 5. Définir l'expérience

Nous entraînons pendant **3 époques**, soit trois passages sur les 1 000 exemples. Le T4 traite un exemple à la fois et accumule les gradients de 16 exemples avant de mettre à jour les adaptateurs.
Nous enregistrons aussi les versions réellement installées pour pouvoir reproduire l'exécution.


In [ ]:
config = dict(
    model_id=MODEL_ID, model_revision=MODEL_REVISION,
    seed=SEED, train_examples=len(train_rows), validation_examples=len(val_rows),
    test_examples=len(test_rows), epochs=3, learning_rate=2e-4,
    lora_rank=16, lora_alpha=32, max_length=512,
    batch_size=1, gradient_accumulation_steps=16,
    gpu=torch.cuda.get_device_name(0), compute_dtype="float16",
    eval_generation_batch_size=1,
    quantization="bitsandbytes NF4 4-bit + double quantization",
    packages={name: version(name) for name in [
        "torch", "transformers", "peft", "bitsandbytes", "accelerate", "datasets",
    ]},
)
(RUN_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(json.dumps(config, indent=2))


## 6. Construire la consigne et la réponse attendue

Le **tokenizer** transforme le texte en nombres compréhensibles par le modèle. Son format de conversation ajoute les marqueurs propres à Gemma.
La consigne contient les 60 noms d'intentions. Nous demandons une seule étiquette et désactivons le mode de raisonnement avec `enable_thinking=False`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer.padding_side = "right"
SYSTEM = (
    "Classe la demande française dans une intention du corpus MASSIVE. "
    "Réponds uniquement avec une étiquette exacte parmi : " + ", ".join(intent_names) + "."
)
(RUN_DIR / "system_prompt.txt").write_text(SYSTEM, encoding="utf-8")


def messages_for(row):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": row["utt"]}]


def prompt_for(row):
    return tokenizer.apply_chat_template(
        messages_for(row), tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )

print("Phrase :", train_rows[0]["utt"])
print("Réponse attendue :", train_rows[0]["intent"])


### Calculer la perte uniquement sur la réponse

Le modèle lit la consigne et la phrase, mais nous voulons lui apprendre **la réponse**. Dans `labels`, la valeur `-100` indique les positions à ignorer dans le calcul de la perte.
Nous vérifions que le début de la conversation complète correspond exactement au prompt. Les exemples de plus de 512 tokens déclenchent une erreur : nous ne tronquons pas silencieusement leur réponse.


In [ ]:
def encode_row(row):
    prefix = tokenizer(prompt_for(row), add_special_tokens=False)["input_ids"]
    complete = tokenizer.apply_chat_template(
        messages_for(row) + [{"role": "assistant", "content": row["intent"]}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )
    ids = tokenizer(complete, add_special_tokens=False)["input_ids"]
    assert ids[:len(prefix)] == prefix, "Le prompt et la conversation ne correspondent pas."
    assert len(ids) <= config["max_length"], f"Exemple trop long : {len(ids)} tokens"
    labels = [-100] * len(prefix) + ids[len(prefix):]
    assert sum(value != -100 for value in labels) >= 2
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_ds = Dataset.from_list([encode_row(row) for row in train_rows])
val_ds = Dataset.from_list([encode_row(row) for row in val_rows])
lengths = [len(ids) for ids in train_ds["input_ids"]]
print("Longueurs des exemples :", min(lengths), "à", max(lengths), "tokens")
supervised = [token for token in train_ds[0]["labels"] if token != -100]
print("Texte sur lequel la perte est calculée :", tokenizer.decode(supervised))


## 7. Charger la base en 4 bits

La quantification **NF4** réduit la mémoire nécessaire aux poids. La double quantification réduit aussi le stockage de ses paramètres auxiliaires.
Les calculs utilisent le fp16. La base est gelée ; ses grandes tables d'embeddings restent en fp16 pour tenir dans la mémoire du T4.


In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=torch.float16, device_map={"": 0},
    attn_implementation="sdpa", quantization_config=quantization,
)
assert getattr(model, "is_loaded_in_4bit", False)
model.requires_grad_(False)
for name, parameter in model.named_parameters():
    if "norm" in name:
        parameter.data = parameter.data.float()


### Ajouter les adaptateurs LoRA

LoRA ajoute de petites matrices entraînables aux projections d'attention et de MLP. Le **rang 16** fixe leur taille.
La règle `target_modules` vise uniquement les couches de langage : nous n'adaptons ni la vision ni l'audio.
Le calcul intermédiaire est recomputé pendant la rétropropagation pour économiser de la mémoire (`gradient_checkpointing`).


In [ ]:
model.config.use_cache = False
model.config.get_text_config().use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

language_targets = (
    r"^model\.language_model\.layers\.\d+\."
    r"(?:self_attn\.(?:q_proj|k_proj|v_proj|o_proj)|mlp\.(?:gate_proj|up_proj|down_proj))$"
)
lora_config = LoraConfig(
    r=config["lora_rank"], lora_alpha=config["lora_alpha"],
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", revision=MODEL_REVISION,
    target_modules=language_targets,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
assert all("lora_" in name for name, p in model.named_parameters() if p.requires_grad)


### Faire un premier calcul et vérifier les gradients

Avant les trois époques, nous testons l'exemple le plus long. Une perte finie et des gradients non nuls montrent que les adaptateurs peuvent apprendre.
Cette vérification ne met pas à jour les poids ; les gradients sont effacés avant l'entraînement.


In [ ]:
collator = DataCollatorForSeq2Seq(
    tokenizer, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100,
)
longest = max(range(len(train_ds)), key=lambda i: len(train_ds[i]["input_ids"]))
probe = {key: value.to("cuda") for key, value in collator([train_ds[longest]]).items()}
model.train()
with torch.autocast("cuda", dtype=torch.float16):
    loss = model(**probe).loss
assert torch.isfinite(loss), "La perte initiale n'est pas finie."
loss.backward()
grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
assert grads and all(torch.isfinite(gradient).all() for gradient in grads)
assert any(gradient.abs().sum() > 0 for gradient in grads), "Aucun gradient utile."
print("Vérification réussie, perte initiale :", float(loss))

model.zero_grad(set_to_none=True)
del probe, loss, grads
gc.collect()
torch.cuda.empty_cache()


## 8. Configurer l'entraînement

À la fin de chaque époque, nous calculons la perte sur la validation et sauvegardons un checkpoint.
`load_best_model_at_end=True` recharge celui dont la perte de validation est la plus faible. **Le test n'intervient pas dans ce choix.**


In [ ]:
args = TrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"), num_train_epochs=config["epochs"],
    per_device_train_batch_size=config["batch_size"], per_device_eval_batch_size=1,
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    learning_rate=config["learning_rate"],
    optim="paged_adamw_8bit", lr_scheduler_type="cosine", warmup_ratio=0.05,
    max_grad_norm=1.0, weight_decay=0.01,
    fp16=True, bf16=False, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    greater_is_better=False, logging_steps=10, report_to="none",
    seed=config["seed"], data_seed=config["seed"], prediction_loss_only=True, dataloader_num_workers=0,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, processing_class=tokenizer,
)


### Lancer les trois époques et sauvegarder le meilleur adaptateur

Cette cellule réalise l'entraînement. Le dossier `adapter/` contient ensuite l'adaptateur retenu et le tokenizer.
Nous conservons également les pertes, la durée et le pic de mémoire GPU allouée par PyTorch.


In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_result = trainer.train()
train_seconds = time.time() - t0
peak_training_vram_gib = torch.cuda.max_memory_allocated() / 2**30

adapter_dir = RUN_DIR / "adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(adapter_dir)
(RUN_DIR / "training_log.json").write_text(
    json.dumps(trainer.state.log_history, indent=2), encoding="utf-8",
)
trainer.state.save_to_json(str(RUN_DIR / "trainer_state.json"))
print("Adaptateur sauvegardé :", adapter_dir)


### Lire les courbes de perte

La courbe bleue mesure la perte pendant l'entraînement ; les points orange mesurent la validation après chaque époque.
Une perte plus faible indique une meilleure prédiction des réponses attendues. Ces courbes ne remplacent pas la mesure finale de classification sur le test.


In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for key, label, color in [
    ("loss", "Entraînement", "#2563eb"), ("eval_loss", "Validation", "#d97706"),
]:
    points = [(entry["epoch"], entry[key]) for entry in logs if key in entry and "epoch" in entry]
    if points:
        epochs, losses = zip(*points)
        ax.plot(epochs, losses, marker="o" if key == "eval_loss" else None, label=label, color=color)
ax.set(xlabel="Époque", ylabel="Perte", title="Gemma 4 E2B — MASSIVE français")
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False)
fig.savefig(RUN_DIR / "pertes.png", dpi=180)
plt.show()


## 9. Générer une intention pour chaque phrase de test

Pour mesurer le comportement réel du modèle, nous lui faisons générer son étiquette sans lui montrer la réponse attendue.
L'évaluation utilise **une phrase à la fois, sans padding**, et une génération déterministe (`do_sample=False`).
Le token `<turn|>` marque la fin de la réponse Gemma. Nous réservons assez de tokens pour la plus longue étiquette.


In [ ]:
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.config.get_text_config().use_cache = True
tokenizer.padding_side = "left"
end_id = tokenizer.convert_tokens_to_ids("<turn|>")
assert end_id != tokenizer.unk_token_id
MAX_NEW_TOKENS = max(len(tokenizer(label, add_special_tokens=False)["input_ids"]) for label in intent_names) + 4
assert all(
    len(tokenizer(prompt_for(row), add_special_tokens=False)["input_ids"]) + MAX_NEW_TOKENS <= config["max_length"]
    for row in test_rows
)


### Définir la prédiction d'une seule phrase

Nous ne gardons que les tokens produits **après** le prompt. La sortie est comparée à l'étiquette exacte ; une réponse libre ou une étiquette inconnue n'est pas corrigée automatiquement.


In [ ]:
@torch.inference_mode()
def predict_intent(row):
    inputs = tokenizer(
        [prompt_for(row)], add_special_tokens=False,
        padding=False, return_tensors="pt",
    ).to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        generated = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            eos_token_id=end_id, pad_token_id=tokenizer.pad_token_id, use_cache=True,
        )
    answer = generated[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(answer, skip_special_tokens=True)[0].strip()


### Mesurer la qualité et sauvegarder les prédictions

Nous calculons trois indicateurs : **accuracy**, la proportion de réponses correctes ; **macro-F1**, qui donne le même poids à chacune des 59 intentions présentes dans le test ; et **invalid_rate**, la proportion de sorties hors des 60 étiquettes autorisées.
Les prédictions et le rapport par intention sont sauvegardés pour pouvoir examiner les erreurs.


In [ ]:
def evaluate_generation(tag):
    predictions = []
    for index, row in enumerate(test_rows):
        predictions.append(predict_intent(row))
        if index % 25 == 0:
            print(f"{tag} : {index + 1}/{len(test_rows)}", flush=True)

    gold = [row["intent"] for row in test_rows]
    supported = sorted(set(gold))
    result = dict(
        accuracy=accuracy_score(gold, predictions),
        macro_f1=f1_score(gold, predictions, labels=supported, average="macro", zero_division=0),
        invalid_rate=sum(pred not in intent_names for pred in predictions) / len(predictions),
        n=len(gold), supported_intents=len(supported),
    )
    with (RUN_DIR / f"predictions_{tag}.jsonl").open("w", encoding="utf-8") as stream:
        for row, prediction in zip(test_rows, predictions):
            stream.write(json.dumps({**row, "prediction": prediction}, ensure_ascii=False) + "\n")
    report = classification_report(
        gold, predictions, labels=supported, output_dict=True, zero_division=0,
    )
    (RUN_DIR / f"report_{tag}.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(tag, json.dumps(result, indent=2))
    return result


### Comparer l'adaptateur et la base sur le même test

Nous évaluons d'abord le meilleur adaptateur, puis désactivons LoRA avec `disable_adapter()` pour retrouver la base quantifiée.
Le prompt, les phrases et la méthode de génération restent les mêmes : nous pouvons ainsi mesurer l'effet de l'adaptation.


In [ ]:
finetuned = evaluate_generation("qlora")
with model.disable_adapter():
    baseline = evaluate_generation("base_4bit")

metrics = dict(
    finetuned=finetuned, baseline=baseline,
    accuracy_gain=finetuned["accuracy"] - baseline["accuracy"],
    training_seconds=train_seconds,
    peak_training_vram_gib=peak_training_vram_gib,
    training=train_result.metrics,
)
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Gain d'accuracy :", f"{100 * metrics['accuracy_gain']:+.2f} points")


## 10. Créer une seule archive avec tous les résultats

L'archive contient l'adaptateur, le tokenizer, le prompt, les données sélectionnées, les métriques, les prédictions et la courbe de perte.
Nous excluons les caches de téléchargement et les checkpoints intermédiaires. **Les poids complets du modèle de base ne sont pas inclus** : ils seront rechargés depuis la révision enregistrée pour utiliser l'adaptateur.


In [ ]:
import zipfile

(adapter_dir / "README.md").write_text(
    "# Gemma 4 E2B — MASSIVE français\n\n"
    "Adaptateur QLoRA pour la classification des intentions. 1000 exemples français, 3 époques, T4.\n"
    f"Base : {MODEL_ID}, révision {MODEL_REVISION}.\n"
    "Charger la base quantifiée puis cet adaptateur avec PeftModel.from_pretrained.\n"
    "Utiliser le tokenizer joint, system_prompt.txt et enable_thinking=False.\n"
    "Données : AmazonScience/massive, FitzGerald et al., 2022, licence CC BY 4.0.\n"
    "https://huggingface.co/datasets/AmazonScience/massive\n",
    encoding="utf-8",
)
archive_path = str(RUN_DIR) + "_resultats.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.rglob("*"):
        relative_path = path.relative_to(RUN_DIR)
        excluded = any(part in relative_path.parts for part in [".dataset_cache", "checkpoints"])
        if path.is_file() and not excluded:
            archive.write(path, relative_path)
print("Entraînement et évaluation terminés. Archive :", archive_path)


### Télécharger avant de fermer Colab

Les fichiers de `/content` sont temporaires. Télécharger maintenant l'archive permet de conserver l'adaptateur et les résultats.
Le script `predict_gemma4_massive.py`, fourni avec les livrables, permettra ensuite de recharger l'adaptateur extrait pour classifier d'autres phrases.


In [ ]:
from google.colab import files

files.download(archive_path)


## Sources

- [Modèle Gemma 4 E2B de Google](https://huggingface.co/google/gemma-4-E2B-it) : E2B désigne environ 2,3 milliards de paramètres effectifs, 5,1 milliards avec les embeddings.
- [Documentation Gemma 4 dans Transformers](https://huggingface.co/docs/transformers/model_doc/gemma4).
- [Guide QLoRA de Google](https://ai.google.dev/gemma/docs/core/huggingface_text_finetune_qlora).
- [Corpus MASSIVE](https://huggingface.co/datasets/AmazonScience/massive) et [dépôt officiel](https://github.com/alexa/massive), FitzGerald et al., licence CC BY 4.0.

Les métriques affichées dans ce notebook proviennent uniquement de l'exécution des cellules d'entraînement et d'évaluation.
